# STATS 425 · Week 4 lab
**Learning from verifiable rewards**  
**TA:** Ali Kaazempur-Mofrad

In Week 2, we updated a model using desired responses. In Week 3, we sampled responses from a model with fixed weights. Today, we combine sampling and training: **sample an answer, check it, and use the reward to update the model.**

This is an introduction to **reinforcement learning with verifiable rewards (RLVR)**. We use a small model that chooses among three answers, so we can inspect the entire calculation and run everything quickly on a central processing unit (CPU). It reads numbers directly; it is not a pretrained language model.

**How this maps to a language model.** The prompt is the context, a response is the action, and a verifier scores that response with a reward. In a language model, a response is a sequence of tokens generated as in Week 3. Here, we shorten the response to a single letter so that every calculation stays visible. We return to this connection in more detail at the end.

| Section | Question we answer |
|---|---|
| 1. Checking answers with a reward | How do we turn correctness into feedback? |
| 2. Sampling from a policy | How does the model choose an answer? |
| 3. Turning rewards into a weight update | Where does the training loss come from? |
| 4. Training on many examples | How do repeated updates change the model? |
| 5. Testing the trained policy | Did the model improve on new examples? |


**Set up the notebook.** The cell below imports the libraries and sets a random seed so the results are repeatable; exact numbers may differ slightly across PyTorch versions.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch>=2.8,<3"])

import copy
import itertools
import math
import random

import torch
from torch import nn  # PyTorch's neural-network module
from torch.distributions import Categorical

SEED = 425
torch.manual_seed(SEED)
torch.set_num_threads(1)

print(f"PyTorch {torch.__version__} | Device: CPU")

PyTorch 2.8.0 | Device: CPU


## 1 · Checking answers with a reward

In supervised fine-tuning, we supply a desired response and increase its probability. In RLVR, the model first produces an answer. A **verifier** checks that answer and assigns a numerical **reward**. Both methods can use known correct answers; the difference is how those answers provide the training signal.

Our task is to choose the largest of three different numbers:

```text
Which number is largest?
A: 2    B: 7    C: 4
Respond with A, B, or C.
```

The model's response is one letter. Here, `B` is correct because it selects 7. We use the reward rule

$$
R(x,y)=
\begin{cases}
1, & \text{if response }y\text{ selects the largest number in }x,\\
0, & \text{otherwise.}
\end{cases}
$$

The prompt is $x$, and the chosen response is $y$. This rule can be checked directly in Python, without a human rating each answer.

**Implement the verifier.** Inside the code, responses `A`, `B`, and `C` have indices 0, 1, and 2. `verify` looks up the selected number and checks whether it is the largest. The code below checks all three possible answers so we can inspect the rule before training.

In [2]:
LETTERS = ("A", "B", "C")
EXAMPLE = (2, 7, 4)


def format_prompt(numbers):
    options = "    ".join(f"{letter}: {int(value)}" for letter, value in zip(LETTERS, numbers))
    return f"Which number is largest?\n{options}\nRespond with A, B, or C."


def verify(numbers, action):
    return float(numbers[action] == max(numbers))


def batch_rewards(numbers, actions):
    # Apply the same verifier to each sampled answer in a batch.
    return torch.tensor([
        verify(row, action)
        for row, action in zip(numbers.tolist(), actions.tolist())
    ], dtype=torch.float32)


print(format_prompt(EXAMPLE))
print("-" * 48)
print(f"{'Response':<12}{'Selected number':<20}{'Reward':>8}")
for action, letter in enumerate(LETTERS):
    print(f"{letter:<12}{EXAMPLE[action]:<20}{verify(EXAMPLE, action):>8.0f}")

Which number is largest?
A: 2    B: 7    C: 4
Respond with A, B, or C.
------------------------------------------------
Response    Selected number       Reward
A           2                          0
B           7                          1
C           4                          0


Only `B` receives reward 1. **Checking an answer does not change the model.** To learn from this feedback, we still need a probability for the sampled answer, a loss, and an optimizer step.

## 2 · Sampling from a policy

In reinforcement learning, the model's distribution over possible actions is called a **policy**. Here, taking an action means choosing an answer. Denote the policy as $\pi_\theta(y\mid x)$: the probability of choosing response $y$ for prompt $x$, with weights $\theta$.

In our task, the model observes one prompt, takes one action, and receives one reward. This setting is called a **contextual bandit**, the simplest reinforcement-learning problem: there is no sequence of later decisions to plan for, so we can focus on how rewards change the policy.

This connects directly to Week 3's logits and softmax. We calculate the answer probabilities in two steps: first compute a score for each answer, then convert those scores into probabilities.

Our small model reads the three numbers as a vector, scaled by 10:

$$
u=\frac{1}{10}(x_A,x_B,x_C)^\top.
$$

For the example $(2,7,4)$, this gives $u=(0.2,0.7,0.4)^\top$. Dividing by 10 keeps the inputs at a convenient scale for training; it does not tell the model which answer is correct.

**Calculate the logits.** A logit is a score for an answer before we convert it into a probability. For each possible answer $j\in\{A,B,C\}$, the model computes

$$
z_j=w_j^\top u
    =w_{jA}u_A+w_{jB}u_B+w_{jC}u_C.
$$

The vector $w_j$ contains the three learned weights used to score answer $j$. Each weight multiplies one input, and we add the results to obtain the logit. Writing all three calculations together gives

$$
\begin{pmatrix}z_A\\z_B\\z_C\end{pmatrix}
=
\underbrace{\begin{pmatrix}
w_{AA}&w_{AB}&w_{AC}\\
w_{BA}&w_{BB}&w_{BC}\\
w_{CA}&w_{CB}&w_{CC}
\end{pmatrix}}_{W}
\begin{pmatrix}u_A\\u_B\\u_C\end{pmatrix},
\qquad z=Wu.
$$

Thus, $W$ is a learned $3\times3$ matrix: nine weights produce three logits. Logits can be positive, negative, or zero; they are not yet probabilities.

**Convert the logits into probabilities using softmax.** For answer $j$,

$$
\pi_\theta(y=j\mid x)
=\frac{\exp(z_j)}{\displaystyle\sum_{k\in\{A,B,C\}}\exp(z_k)}
=\frac{e^{z_j}}{e^{z_A}+e^{z_B}+e^{z_C}}.
$$

Exponentiating makes each term positive. Dividing by their sum makes the three probabilities add up to 1. An answer with a larger logit receives a larger probability.

We start all nine weights at zero, so all three logits are zero. Substituting into softmax gives the same probability for every answer:

$$
\pi_\theta(y=j\mid x)=\frac{e^0}{e^0+e^0+e^0}=\frac{1}{3}.
$$

`Categorical(logits=...)` uses these softmax probabilities to sample an answer. We use the full distribution, with temperature 1 and no top-k or top-p filtering.

**Sample eight answers to the same prompt.** These are eight independent draws from the starting policy. Sampling changes which answer is chosen; it does not update the weights.

In [3]:
def make_policy():
    model = nn.Linear(3, 3, bias=False)
    nn.init.zeros_(model.weight)
    return model


def answer_distribution(model, numbers):
    logits = model(numbers.float() / 10.0)
    return Categorical(logits=logits)


torch.manual_seed(SEED)
initial_policy = make_policy()
demo_numbers = torch.tensor(EXAMPLE, dtype=torch.float32)
demo_distribution = answer_distribution(initial_policy, demo_numbers)
sampled_actions = demo_distribution.sample((8,))

print(format_prompt(EXAMPLE))
print("-" * 60)
print("STARTING PROBABILITIES")
for letter, probability in zip(LETTERS, demo_distribution.probs.detach().tolist()):
    print(f"P({letter}) = {probability:.4f}")
print("-" * 60)
print("EIGHT SAMPLED RESPONSES")
print("  ".join(LETTERS[action] for action in sampled_actions.tolist()))

Which number is largest?
A: 2    B: 7    C: 4
Respond with A, B, or C.
------------------------------------------------------------
STARTING PROBABILITIES
P(A) = 0.3333
P(B) = 0.3333
P(C) = 0.3333
------------------------------------------------------------
EIGHT SAMPLED RESPONSES
B  C  B  A  C  C  C  B


Each answer has the same probability, but eight random draws need not contain equal numbers of `A`, `B`, and `C`. We will now score these exact samples and use them for one update.

## 3 · Turning rewards into a weight update

Our goal is to increase **expected reward**: the average reward we would obtain by sampling from the policy. We use a basic policy-gradient method called **REINFORCE** ("REward Increment = Nonnegative Factor × Offset Reinforcement × Characteristic Eligibility"). The name comes from the original update rule; here, its practical role is to use sampled rewards to guide the weight updates. [Williams, 1992](https://gwern.net/doc/reinforcement-learning/model-free/1992-williams.pdf).

Keep two ideas separate. **RLVR describes where the reward comes from:** an automatic check of the answer. **REINFORCE describes how the model learns from that reward.** Next week, we improve the update itself; in Week 6, we change the reward.

The **negative log-likelihood (NLL)** of a sampled answer is $-\ln\pi_\theta(y_i\mid x_i)$. It is smaller when the model assigns that answer a higher probability. As in Weeks 1–3, $\ln$ denotes the natural logarithm.

For a batch of $B$ sampled answers, we multiply each answer's NLL by its reward and average the results. The loss used for an update is

$$
L(\theta)=-\frac{1}{B}\sum_{i=1}^{B}
\underbrace{r_i}_{\text{reward}}\ \ 
\underbrace{\ln\pi_\theta(y_i\mid x_i)}_{\text{log-probability of the sampled answer}},
\qquad r_i=R(x_i,y_i).
$$

This is the same negative log-probability we saw in Weeks 1–2, now weighted by the reward. Here $B$ denotes batch size.

- **Reward 1:** the sample contributes $-\ln\pi_\theta(y_i\mid x_i)$. The update encourages the model to make that answer more likely.
- **Reward 0:** the sample contributes zero and supplies no gradient in this simple version. Other rewarded samples can still change its probability because the weights are shared and the probabilities must sum to one.

For our starting probabilities, a correct sample contributes $-\ln(1/3)\approx1.0986$; an incorrect sample contributes 0. We add these contributions and divide by **all eight samples**, including those with reward 0.

**Why use this loss?** If $J(\theta)$ is expected reward, the policy-gradient identity gives

$$
J(\theta)=\mathbb{E}_{x}\mathbb{E}_{y\sim\pi_\theta(\cdot\mid x)}[R(x,y)],
\qquad
\nabla_\theta J(\theta)=
\mathbb{E}[R(x,y)\nabla_\theta\ln\pi_\theta(y\mid x)].
$$

The negative gradient of our sampled loss estimates this direction. Gradient descent on the loss therefore estimates gradient ascent on expected reward. The verifier's reward is treated as a fixed number: **we differentiate the log-probability, not the correctness check.**

**Calculate the gradients from the loss.** A gradient tells us how the loss changes when we change a logit or weight. We first differentiate with respect to the logits, then use the input values to obtain the weight gradients.

All eight answers in this example are sampled for the same prompt. They therefore share the same logits $z$ and probabilities $p_j=\pi_\theta(y=j\mid x)$. We hold the sampled answers and their rewards fixed when calculating this update.

**1. Differentiate the loss with respect to each logit.** For one sampled answer $y_i$, substitute softmax into its negative log-likelihood:

$$
-\ln p_{y_i}=-z_{y_i}+\ln\left(\sum_{k\in\{A,B,C\}}e^{z_k}\right).
$$

The derivative of the first term is $-1$ if $j=y_i$ and 0 otherwise. The derivative of the second term is $e^{z_j}/\sum_k e^{z_k}=p_j$. Combining these derivatives, multiplying by each reward, and averaging gives

$$
g_j\equiv\frac{\partial L}{\partial z_j}
=\frac{1}{B}\sum_{i=1}^{B}r_i\left(p_j-\mathbf{1}\{y_i=j\}\right)
=\frac{p_j\sum_i r_i-\sum_i r_i\mathbf{1}\{y_i=j\}}{B}.
$$

The indicator $\mathbf{1}\{y_i=j\}$ is 1 when sample $i$ chose answer $j$, and 0 otherwise. The last expression makes the arithmetic easy to follow: multiply $p_j$ by the total reward, subtract the reward contributed by samples that chose $j$, then divide by the batch size.

For the displayed batch, three of eight answers are correct, all three choose `B`, and every starting probability is $1/3$. Thus,

$$
g_A=\frac{3(1/3)-0}{8}=0.125,
\qquad
g_B=\frac{3(1/3)-3}{8}=-0.25,
\qquad
g_C=\frac{3(1/3)-0}{8}=0.125.
$$

The five incorrect samples contribute zero because their rewards are zero. These numbers use the batch shown in the saved output; the code recalculates them from the actual samples each time.

**2. Use the chain rule to obtain the weight gradients.** In Section 2, we wrote $z_j=\sum_k w_{jk}u_k$. Since $\partial z_j/\partial w_{jk}=u_k$,

$$
\frac{\partial L}{\partial w_{jk}}
=\frac{\partial L}{\partial z_j}\frac{\partial z_j}{\partial w_{jk}}
=g_j u_k.
$$

Here $j$ identifies the answer being scored and $k$ identifies an input position. Multiply each logit gradient by $u=(0.2,0.7,0.4)^\top$ to get the three weight gradients in that row:

$$
\nabla_W L=
\begin{pmatrix}
0.125(0.2)&0.125(0.7)&0.125(0.4)\\
-0.25(0.2)&-0.25(0.7)&-0.25(0.4)\\
0.125(0.2)&0.125(0.7)&0.125(0.4)
\end{pmatrix}
=
\begin{pmatrix}
0.025&0.0875&0.05\\
-0.05&-0.175&-0.10\\
0.025&0.0875&0.05
\end{pmatrix}.
$$

These are the nine numbers used in the weight update. `loss.backward()` calculates the same derivatives automatically. The code below also computes the formulas above so we can compare the two results. Tiny differences can occur because computers store numbers with limited precision.

**Calculate the loss and take one step.** We use a copy of the starting policy so this example can be rerun from the same weights. Stochastic gradient descent (SGD) with learning rate 1 makes the probability change easy to see. This large step is just for the worked example.

**Follow the calculation from weights to probabilities.** PyTorch calculates a gradient for each weight: how the loss changes with that weight. SGD uses these gradients to update the weights. We then calculate new logits and apply softmax, using the equations from Section 2:

$$
W_{\mathrm{new}}=W_{\mathrm{old}}-\eta\nabla_W L,
\qquad
z_{\mathrm{new}}=W_{\mathrm{new}}u,
\qquad
p_j^{\mathrm{new}}=
\frac{\exp(z_j^{\mathrm{new}})}{\sum_{k\in\{A,B,C\}}\exp(z_k^{\mathrm{new}})}.
$$

Here $\eta=1$ is the learning rate, and $\nabla_W L$ contains the nine weight gradients. The output shows the logit gradients, the weight gradients, the subtraction for each row of weights, the three new logits, and the softmax division that gives each updated probability.


In [4]:
step_policy = copy.deepcopy(initial_policy)
before_weights = step_policy.weight.detach().clone()
scaled_inputs = demo_numbers / 10.0
distribution = answer_distribution(step_policy, demo_numbers)
before_probabilities = distribution.probs.detach().clone()

repeated_numbers = demo_numbers.repeat(len(sampled_actions), 1)
rewards = batch_rewards(repeated_numbers, sampled_actions)
log_probs = distribution.log_prob(sampled_actions)
negative_log_probs = -log_probs
loss_terms = rewards * negative_log_probs
loss = loss_terms.mean()

print("NLL = negative log-likelihood = -log(probability).")
print(f"{'Draw':<7}{'Response':<11}{'Reward':>8}{'Probability':>14}{'-log(prob)':>14}{'Reward x NLL':>16}")
print("-" * 70)
for i, action in enumerate(sampled_actions.tolist()):
    print(f"{i + 1:<7}{LETTERS[action]:<11}{rewards[i].item():>8.0f}"
          f"{log_probs[i].exp().item():>14.4f}{negative_log_probs[i].item():>14.4f}"
          f"{loss_terms[i].item():>16.4f}")

print("-" * 70)
print(f"Correct samples: {int(rewards.sum())} of {len(rewards)}")
print(f"Mean reward = {rewards.mean().item():.4f}")
print(f"Training loss = sum of contributions / number of samples")
print(f"              = {loss_terms.sum().item():.4f} / {len(loss_terms)} = {loss.item():.4f}")

step_learning_rate = 1.0
optimizer = torch.optim.SGD(step_policy.parameters(), lr=step_learning_rate)
optimizer.zero_grad()
loss.backward()
weight_gradients = step_policy.weight.grad.detach().clone()

# Calculate the same gradients directly from the formulas in the lesson.
total_reward = rewards.sum()
reward_for_each_answer = torch.stack([
    rewards[sampled_actions == action].sum() for action in range(len(LETTERS))
])
manual_logit_gradients = (
    total_reward * before_probabilities - reward_for_each_answer
) / len(sampled_actions)
manual_weight_gradients = manual_logit_gradients[:, None] * scaled_inputs[None, :]
gradient_difference = (manual_weight_gradients - weight_gradients).abs().max().item()

optimizer.step()

with torch.no_grad():
    updated_weights = step_policy.weight.detach().clone()
    updated_logits = step_policy(scaled_inputs)
    after_probabilities = torch.softmax(updated_logits, dim=0)
    exponentials = torch.exp(updated_logits)
    softmax_denominator = exponentials.sum()


def format_vector(values):
    return "(" + ", ".join(f"{value.item():.4f}" for value in values) + ")"


print("\n" + "=" * 70)
print("1. CALCULATE THE GRADIENTS")
print("Logit gradient = (total reward x probability - reward for that answer) / batch size")
for letter, probability, reward_for_answer, gradient in zip(
    LETTERS, before_probabilities, reward_for_each_answer, manual_logit_gradients
):
    print(f"g_{letter} = ({total_reward.item():.0f} x {probability.item():.4f}"
          f" - {reward_for_answer.item():.0f}) / {len(sampled_actions)} = {gradient.item():.4f}")

print(f"\nInput u = {format_vector(scaled_inputs)}")
print("Weight gradients for each answer = its logit gradient x the input")
for letter, gradient, row in zip(LETTERS, manual_logit_gradients, manual_weight_gradients):
    print(f"{letter}: {gradient.item():.4f} x {format_vector(scaled_inputs)} = {format_vector(row)}")
print(f"Largest difference from PyTorch's weight gradients: {gradient_difference:.8f}")

print("\n" + "-" * 70)
print("2. UPDATE THE WEIGHTS")
print(f"New weights = old weights - {step_learning_rate:.1f} x gradients")
print("Each row lists the weights for inputs A, B, and C, in that order.")
for letter, old, gradient, new in zip(LETTERS, before_weights, weight_gradients, updated_weights):
    print(f"{letter}: {format_vector(old)} - {step_learning_rate:.1f} x {format_vector(gradient)}")
    print(f"   = {format_vector(new)}")

print("\n" + "-" * 70)
print("3. CALCULATE THE NEW LOGITS")
print(f"Input u = {format_vector(scaled_inputs)}")
print("Each logit = updated weights dotted with the input.")
for letter, weights, logit in zip(LETTERS, updated_weights, updated_logits):
    terms = " + ".join(
        f"({weight.item():.4f} x {value.item():.1f})"
        for weight, value in zip(weights, scaled_inputs)
    )
    print(f"z_{letter} = {terms} = {logit.item():.5f}")

print("\n" + "-" * 70)
print("4. APPLY SOFTMAX")
print("Sum of exponentials = " + " + ".join(f"exp({z.item():.5f})" for z in updated_logits))
print("                    = " + " + ".join(f"{value.item():.6f}" for value in exponentials))
print(f"                    = {softmax_denominator.item():.6f}")

print("\nPROBABILITIES AFTER ONE UPDATE")
print(f"{'Response':<10}{'Before':>10}{'Calculation: exp(logit) / sum':>34}{'After':>12}")
for letter, before, numerator, after in zip(LETTERS, before_probabilities, exponentials, after_probabilities):
    calculation = f"{numerator.item():.6f} / {softmax_denominator.item():.6f}"
    print(f"{letter:<10}{before.item():>10.4f}{calculation:>34}{after.item():>12.4f}")
print("=" * 70)

NLL = negative log-likelihood = -log(probability).
Draw   Response     Reward   Probability    -log(prob)    Reward x NLL
----------------------------------------------------------------------
1      B                 1        0.3333        1.0986          1.0986
2      C                 0        0.3333        1.0986          0.0000
3      B                 1        0.3333        1.0986          1.0986
4      A                 0        0.3333        1.0986          0.0000
5      C                 0        0.3333        1.0986          0.0000
6      C                 0        0.3333        1.0986          0.0000
7      C                 0        0.3333        1.0986          0.0000
8      B                 1        0.3333        1.0986          1.0986
----------------------------------------------------------------------
Correct samples: 3 of 8
Mean reward = 0.3750
Training loss = sum of contributions / number of samples
              = 3.2958 / 8 = 0.4120

1. CALCULATE THE GRADIENTS
Lo

The rewarded answer is `B`. When the batch contains at least one `B`, this update increases its probability for the example. The probabilities of `A` and `C` fall as probability shifts toward `B`.

## 4 · Training on many examples

One update helps with one prompt. To learn a rule that works across prompts, we now repeat the same process with many examples.

**Prepare the training and test sets.** We use three different integers from 1 through 9. There are 84 unordered groups of three numbers. We use 64 groups for training and reserve 20 groups as a test set to measure how well the trained model performs on new examples. Each group appears in all six orders, giving 384 prompts in the training set and 120 prompts in the test set.

Keeping an entire group in one set means the test set cannot contain a rearrangement of a training prompt. It still uses the same number range, so this checks new combinations within this small task.

In [5]:
groups = list(itertools.combinations(range(1, 10), 3))
random.Random(SEED).shuffle(groups)
train_groups, test_groups = groups[:64], groups[64:]


def expand_orders(groups):
    return torch.tensor([
        ordering
        for group in groups
        for ordering in itertools.permutations(group)
    ], dtype=torch.float32)


train_numbers = expand_orders(train_groups)
test_numbers = expand_orders(test_groups)

print(f"Training set: {len(train_groups)} groups x 6 orders = {len(train_numbers)} prompts")
print(f"Test set: {len(test_groups)} groups x 6 orders = {len(test_numbers)} prompts")
print("-" * 60)
print("EXAMPLE TRAINING PROMPT")
print(format_prompt(train_numbers[0].tolist()))

Training set: 64 groups x 6 orders = 384 prompts
Test set: 20 groups x 6 orders = 120 prompts
------------------------------------------------------------
EXAMPLE TRAINING PROMPT
Which number is largest?
A: 2    B: 3    C: 9
Respond with A, B, or C.


**Repeat the sample → verify → update cycle.** We start again from the uniform policy so that our before-and-after comparison has a clear starting point.

We choose a **batch size of 48**. At each update, we draw 48 training prompts at random, allowing repeats, and sample one answer for each prompt. We then calculate the loss and update the weights. The next update uses fresh answers sampled from the updated policy. This is a key difference from supervised fine-tuning: the training data are not fixed in advance, because the model's own current behavior determines which answers it sees and learns from.

The four lines that define the learning signal are:

```python
distribution = answer_distribution(policy, batch)
actions = distribution.sample()
rewards = batch_rewards(batch, actions)
loss = -(rewards * distribution.log_prob(actions)).mean()
```

The optimizer steps are the same as in Week 2. Here we use adaptive moment estimation (Adam) with learning rate 0.05 for 400 updates. These settings suit this nine-weight model; they are not suggested settings for fine-tuning a language model. Because we draw prompts with replacement, we count updates rather than full passes through the data.

The output reports the fraction of sampled training answers that were correct in each block of 50 updates. Expect improvement overall, with some variation because the prompts and answers are sampled.

In [6]:
torch.manual_seed(SEED)
policy = copy.deepcopy(initial_policy)
starting_weights = policy.weight.detach().clone()
optimizer = torch.optim.Adam(policy.parameters(), lr=0.05)

BATCH_SIZE = 48
NUM_UPDATES = 400
REPORT_EVERY = 50
training_history = []
correct_in_block = 0
answers_in_block = 0

print(f"{'Updates':<16}{'Sampled training answers correct':>34}")
print("-" * 52)

for update in range(1, NUM_UPDATES + 1):
    indices = torch.randint(len(train_numbers), (BATCH_SIZE,))
    batch = train_numbers[indices]

    distribution = answer_distribution(policy, batch)
    actions = distribution.sample()
    rewards = batch_rewards(batch, actions)
    loss = -(rewards * distribution.log_prob(actions)).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    correct_in_block += int(rewards.sum().item())
    answers_in_block += len(rewards)
    training_history.append({"update": update, "mean_reward": rewards.mean().item()})

    if update % REPORT_EVERY == 0:
        first_update = update - REPORT_EVERY + 1
        fraction_correct = correct_in_block / answers_in_block
        print(f"{f'{first_update}-{update}':<16}{fraction_correct:>33.1%}")
        correct_in_block = 0
        answers_in_block = 0

weight_change = (policy.weight.detach() - starting_weights).abs().max().item()
print(f"\nLargest absolute change in a weight: {weight_change:.4f}")

Updates           Sampled training answers correct
----------------------------------------------------
1-50                                        45.5%
51-100                                      65.5%
101-150                                     78.0%
151-200                                     79.0%
201-250                                     84.7%
251-300                                     86.1%
301-350                                     87.5%
351-400                                     88.8%

Largest absolute change in a weight: 7.5751


These are rewards from answers sampled **during training**, while the weights were changing. They indicate how training progressed, but they do not yet tell us how the final model performs on the test set. The weight change confirms that optimization changed the model.

Notice that the training loss uses only the sampled answers, their log-probabilities, and their rewards. We do not pass the correct answer as a cross-entropy target, as we did in supervised fine-tuning.

## 5 · Testing the trained policy

We now fix the weights and compare performance on the training and test sets. Within each set, we measure the starting and final policies on the same prompts. We report two different quantities:

| Quantity | How we calculate it | What it tells us |
|---|---|---|
| Expected reward | Average probability assigned to a correct answer | How often an answer sampled from the full policy would be correct, on average |
| Greedy accuracy | Fraction of prompts where the most likely answer is correct | How often we would be correct if we always chose the most likely answer |

For a single prompt, expected reward is

$$
\mathbb{E}[R\mid x]=\sum_{y\in\{A,B,C\}}\pi_\theta(y\mid x)R(x,y).
$$

Since exactly one answer receives reward 1, this sum is just the probability of that answer. For example, if the probabilities are $(0.10,0.70,0.20)$ and `B` is correct, expected reward is $0.70$. Greedy decoding chooses `B` every time for that prompt, so its greedy accuracy is 100%.

With only three possible answers, we can calculate expected reward exactly instead of estimating it from another random sample. This calculation measures performance after training; training used sampled answers only. We average both metrics over all prompts in each set.

In [7]:
@torch.no_grad()
def measure_performance(model, numbers):
    probabilities = answer_distribution(model, numbers).probs
    rewards_for_all_answers = torch.tensor([
        [verify(row, action) for action in range(3)]
        for row in numbers.tolist()
    ], dtype=torch.float32)

    expected_reward = (probabilities * rewards_for_all_answers).sum(dim=1).mean().item()
    greedy_actions = probabilities.argmax(dim=1)
    greedy_accuracy = batch_rewards(numbers, greedy_actions).mean().item()
    return {"expected_reward": expected_reward, "greedy_accuracy": greedy_accuracy}


performance_results = {}
for split_name, numbers in [("Training set", train_numbers), ("Test set", test_numbers)]:
    before = measure_performance(initial_policy, numbers)
    after = measure_performance(policy, numbers)
    performance_results[split_name] = {"before": before, "after": after}

    print("=" * 62)
    print(f"{split_name.upper()} | {len(numbers)} prompts")
    print(f"{'Metric':<38}{'Before':>12}{'After':>12}")
    print(f"{'Expected reward':<38}{before['expected_reward']:>12.4f}{after['expected_reward']:>12.4f}")
    print(f"{'Greedy accuracy':<38}{before['greedy_accuracy']:>12.1%}{after['greedy_accuracy']:>12.1%}")
print("=" * 62)

TRAINING SET | 384 prompts
Metric                                      Before       After
Expected reward                             0.3333      0.8973
Greedy accuracy                              33.3%      100.0%
TEST SET | 120 prompts
Metric                                      Before       After
Expected reward                             0.3333      0.9084
Greedy accuracy                              33.3%      100.0%


The starting policy has expected reward $1/3$. Its three probabilities are tied, so `argmax` chooses the first option, `A`. Because each option is correct equally often in both sets, its starting greedy accuracy is also one-third.

After training, compare the test-set results with the starting policy's results. Improvement shows that the model learned something useful for new combinations of these numbers. It does not establish broad mathematical reasoning ability: this is a simple comparison task with direct numerical inputs.

Greedy accuracy can reach 100% while expected reward remains below 1. The correct answer may be the most likely on every prompt, while incorrect answers still have some probability and can be sampled.

**Inspect an individual prompt.** Set `YOUR_NUMBERS` to three different integers from 1 through 9. Compare the starting and final probabilities, then predict the greedy answer before reading the last line. This checks the model's predictions without updating its weights.

In [8]:
YOUR_NUMBERS = (3, 8, 6)

if (len(YOUR_NUMBERS) != 3 or len(set(YOUR_NUMBERS)) != 3
        or any(type(value) is not int or not 1 <= value <= 9 for value in YOUR_NUMBERS)):
    raise ValueError("Choose three different integers from 1 through 9.")

numbers = torch.tensor(YOUR_NUMBERS, dtype=torch.float32)
with torch.no_grad():
    before = answer_distribution(initial_policy, numbers).probs
    after = answer_distribution(policy, numbers).probs

print("=" * 62)
print("YOUR PROMPT")
print(format_prompt(YOUR_NUMBERS))
print("-" * 62)
print(f"{'Response':<12}{'Number':>10}{'Before':>12}{'After':>12}{'Reward':>10}")
for action, letter in enumerate(LETTERS):
    print(f"{letter:<12}{YOUR_NUMBERS[action]:>10}{before[action].item():>12.4f}"
          f"{after[action].item():>12.4f}{verify(YOUR_NUMBERS, action):>10.0f}")
print("-" * 62)
print(f"Final greedy answer: {LETTERS[after.argmax().item()]}")
print("=" * 62)

YOUR PROMPT
Which number is largest?
A: 3    B: 8    C: 6
Respond with A, B, or C.
--------------------------------------------------------------
Response        Number      Before       After    Reward
A                    3      0.3333      0.0015         0
B                    8      0.3333      0.9238         1
C                    6      0.3333      0.0747         0
--------------------------------------------------------------
Final greedy answer: B


**How does this connect to a language model?**

Our policy chooses one letter in one step. A language model produces a response token by token, as in Week 3. For response tokens $y_1,\ldots,y_T$, its log-probability is

$$
\ln\pi_\theta(y\mid x)=\sum_{t=1}^{T}\ln p_\theta(y_t\mid x,y_{<t}).
$$

A verifier can score the completed response: for example, check its final numerical answer or run tests on generated code. The basic REINFORCE loss then multiplies this summed response log-probability by the reward. The prompt supplies context; its tokens are not included in this response sum.

Reinforcement learning for language models usually starts from a pretrained or supervised fine-tuned model, which can already generate useful responses. Our uniform starting policy is practical because there are only three possible answers. This lesson demonstrates the basic reward-based update; it does not implement a full training method for large language models (LLMs), such as group relative policy optimization (GRPO).

## Recap

We followed one training cycle from sampled answers to updated weights:

**Prompt → sampled answer → verifier reward → reward-weighted log-probability → gradient → weight update**

Sampling supplies candidate answers. A verifier assigns rewards. Reward-weighted log-probabilities supply a gradient, and the optimizer changes the weights. We then tested the fixed policy on the test set to check whether its behavior improved.

### Check your understanding

1. If a whole batch receives reward 0, what happens to the gradient in this implementation?
2. Why can greedy accuracy be 100% while the chance of a correct sampled answer is lower?
3. What would the model learn if a faulty verifier rewarded `A` regardless of the numbers?

<details><summary>Answers · open after discussing</summary>

1. Every term in the loss is zero, so the gradient is zero. With plain SGD, the weights would not change; an optimizer with momentum, such as Adam, can still move them using past gradients.
2. Greedy decoding always chooses the most likely answer. Sampling can still draw an incorrect answer with whatever probability remains, so the chance of a correct sampled answer equals the probability of the correct answer, which can be below 1.
3. It would learn to choose `A` regardless of the numbers. The model optimizes the reward it receives, so a faulty verifier teaches the wrong behavior.

</details>


### Next week: baselines and group-relative advantages

In this lab, an answer with reward 0 contributed nothing to the update, and the update could change a lot from one batch to the next. Next week, we keep the verifiable reward and improve the update. We subtract a **baseline** from each reward, so answers that do better than expected become more likely and answers that do worse become less likely. This lowers the variance of the update without changing its expected direction.

We then sample a **group** of responses for each prompt and compare each reward with the group's average, as in group relative policy optimization (**GRPO**), a method used to train language models that reason step by step. In Week 6, we change the reward itself so that it also scores the model's reported confidence.

## References and further reading

- [OpenAI Spinning Up: Introduction to policy optimization](https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html): policy gradients and reward-weighted log-probabilities.
- [PyTorch: Categorical distributions](https://docs.pytorch.org/docs/2.8/distributions.html#categorical): sampling and calculating log-probabilities.
- [Shao et al. (2024): DeepSeekMath](https://arxiv.org/abs/2402.03300): introduces group relative policy optimization (GRPO), previewed for next week.